# HT206B1 manual alignment: U2–U5
Four slices; three adjacent pairs (U2–U3, U3–U4, U4–U5). U1 is excluded. Feature cutoffs are 0.2 for both stages; existing matching results are reused.


In [ ]:
from pathlib import Path
import os, sys, json, time, hashlib, gzip, gc
os.environ.setdefault("NUMBA_CACHE_DIR", "/tmp/raftup_numba")
os.environ.setdefault("MPLCONFIGDIR", "/tmp/raftup_mpl")
for key in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(key, "4")
import numpy as np
import pandas as pd
import anndata as ad
import matplotlib.pyplot as plt
from PIL import Image
from scipy import sparse
from scipy.io import mmread
from scipy.spatial import cKDTree
from IPython.display import display
ROOT = Path(os.environ.get("RAFTUP_EXPERIMENT_ROOT", "/Users/salovjade/Library/CloudStorage/Dropbox/raftup_revision_828"))
DATA = ROOT / "all_exp/non_brain/HT206B1"
OUT = DATA / "raftup_manual_fc020_v1"
AUTO_OUT = DATA / "raftup_adjacent_v1"
SRC = Path(os.environ.get("RAFTUP_SRC", "/Users/salovjade/Library/CloudStorage/Dropbox/boatwhere/raftup_repo/src"))
sys.path.insert(0, str(SRC))
import torch
from raftup._cutoff_selection import compute_scanit_cost
from raftup._manual_alignment import run_manual_alignment
from raftup._metrics_two_gpr import GPR_original
import raftup._cutoff_selection as cs
assert Path(cs.__file__).resolve() == (SRC / "raftup/_cutoff_selection.py").resolve()
torch.set_num_threads(4)
SECTIONS = ["U2", "U3", "U4", "U5"]
PAIRS = list(zip(SECTIONS[:-1], SECTIONS[1:]))
DEVICE = "cpu"  # Set "cuda" on a compatible Linux GPU environment if desired.
SCANIT = dict(n_epoch=3500, seed=0, device=DEVICE)
GPR_RADIUS_UM = 300.0
# Optional external labels, columns: section, barcode, label. Not used for alignment.
ANNOTATIONS = None
OUT.mkdir(parents=True, exist_ok=True)
manifest = json.loads((DATA / "download_manifest.json").read_text())
def source_file(section, suffix):
    row = next(r for r in manifest if r["section"] == section and r["filename"].endswith(suffix))
    p = DATA / row["filename"]
    if not p.exists(): p = DATA / row["relative_path"]
    assert p.is_file(), p
    return p
def sha(p):
    h = hashlib.sha256()
    with Path(p).open("rb") as f:
        for chunk in iter(lambda: f.read(1024*1024), b""): h.update(chunk)
    return h.hexdigest()
def write_json(p, value):
    p = Path(p); temp = p.with_suffix(p.suffix + ".tmp")
    temp.write_text(json.dumps(value, indent=2, allow_nan=False)); temp.replace(p)
print("Python:", sys.executable, "\nLocal source:", cs.__file__, "\nOutputs:", OUT)
MANUAL = dict(downsampling_ratio=0.067, feature_cutoff_downsample=0.2,
              spatial_cutoff_downsample_um=400, feature_cutoff_recovery=0.2,
              spatial_cutoff_recovery_um=200, k=20, random_state=42,
              min_downsampled_points=150, max_downsampled_points=800)


## 1. Load and calibrate U2–U5


In [ ]:
slices = {}; calibration = []
for section in SECTIONS:
    genes = pd.read_csv(source_file(section, "features.tsv.gz"), sep="\t", header=None)
    barcodes = pd.read_csv(source_file(section, "barcodes.tsv.gz"), header=None)[0].astype(str)
    pos = pd.read_csv(source_file(section, "tissue_positions_list.csv"), header=None,
                      names=["barcode", "in_tissue", "array_row", "array_col", "pixel_y", "pixel_x"]).set_index("barcode")
    assert pos.index.is_unique and barcodes.is_unique and genes[0].is_unique
    xy_all = pos[["pixel_x", "pixel_y"]].to_numpy(float)
    nn = cKDTree(xy_all).query(xy_all, k=2)[0][:, 1]
    pixels_per_100um = float(np.median(nn))
    assert pixels_per_100um > 0 and np.std(nn)/np.mean(nn) < .05, "Unexpected grid geometry"
    ordered = pos.loc[barcodes].copy(); keep = ordered.in_tissue.eq(1).to_numpy()
    with gzip.open(source_file(section, "matrix.mtx.gz"), "rb") as f:
        counts = mmread(f).T.tocsr()
    assert counts.shape == (len(barcodes), len(genes))
    assert np.isfinite(counts.data).all() and (counts.data >= 0).all()
    obs = ordered.iloc[np.flatnonzero(keep)].copy(); obs.index.name = "barcode"
    var = pd.DataFrame({"gene_symbol": genes[1].astype(str).to_numpy()}, index=genes[0].astype(str).to_numpy())
    a = ad.AnnData(counts[keep], obs=obs, var=var)
    a.obsm["spatial_pixels"] = obs[["pixel_x", "pixel_y"]].to_numpy(float)
    a.obsm["spatial"] = a.obsm["spatial_pixels"] * 100 / pixels_per_100um
    a.uns["coordinate_units"] = "um"
    a.uns["pixels_per_100um"] = pixels_per_100um
    slices[section] = a
    calibration.append(dict(section=section,spots=a.n_obs,genes=a.n_vars,pixels_per_100um=pixels_per_100um,grid_spacing_cv=float(np.std(nn)/np.mean(nn))))
    del counts
common = slices[SECTIONS[0]].var_names
for section in SECTIONS[1:]: common = common[common.isin(slices[section].var_names)]
for section in SECTIONS: slices[section] = slices[section][:, common].copy()
if ANNOTATIONS is not None:
    labels = pd.read_csv(ANNOTATIONS, dtype=str)
    assert not labels.duplicated(["section", "barcode"]).any()
    for section, a in slices.items():
        lookup = labels[labels.section == section].set_index("barcode").label
        a.obs["evaluation_label"] = lookup.reindex(a.obs_names)
calibration = pd.DataFrame(calibration)
calibration.to_csv(OUT / "calibration.csv", index=False)
display(calibration)
fig, axes = plt.subplots(1, len(SECTIONS), figsize=(4*len(SECTIONS), 4))
for ax, section in zip(axes, SECTIONS):
    a = slices[section]; scale = json.loads(source_file(section, "scalefactors_json.json").read_text())["tissue_lowres_scalef"]
    ax.imshow(Image.open(source_file(section, "tissue_lowres_image.png")))
    xy = a.obsm["spatial_pixels"] * scale
    ax.scatter(xy[:, 0], xy[:, 1], s=1, c="cyan", alpha=.3)
    ax.set_title(section); ax.axis("off")
fig.tight_layout(); fig.savefig(OUT / "input_tissue_overlay.png", dpi=200); plt.show()

## 2. Evaluation and rigid-transform helpers

LAA uses label agreement for covered-row argmax matches, excluding missing labels. Rigid transforms use all alignment weights, allow rotation and translation only, and are used for visualization. They do not define a ground truth.

In [ ]:
def scores_for(P, a, b):
    covered_a = P.sum(axis=1) > 0; covered_b = P.sum(axis=0) > 0
    laa = None; valid_count = 0
    if "evaluation_label" in a.obs and "evaluation_label" in b.obs and covered_a.any():
        rows = np.flatnonzero(covered_a); cols = P[rows].argmax(axis=1)
        la = a.obs.evaluation_label.iloc[rows]; lb = b.obs.evaluation_label.iloc[cols]
        valid = la.notna().to_numpy() & lb.notna().to_numpy()
        valid_count = int(valid.sum())
        if valid_count: laa = float(np.mean(la.to_numpy()[valid] == lb.to_numpy()[valid]))
    gpr = float(GPR_original(P.copy(), a.obsm["spatial"], b.obsm["spatial"], dis_cut=GPR_RADIUS_UM)) if covered_a.any() else None
    return dict(LAA=laa, GPR300_um=gpr if gpr is None or np.isfinite(gpr) else None,
                coverage_A=float(covered_a.mean()), coverage_B=float(covered_b.mean()),
                valid_label_rows=valid_count, LAA_note="Independent labels not supplied" if ANNOTATIONS is None else "Covered-row argmax label agreement")
def rigid_b_to_a(P, x, y):
    mass = float(P.sum())
    if mass <= 0: raise ValueError("Cannot fit a transform to an empty alignment")
    wa = P.sum(1); wb = P.sum(0)
    mx = wa @ x / mass; my = wb @ y / mass
    H = (y-my).T @ P.T @ (x-mx)
    u, _, vt = np.linalg.svd(H)
    fix = np.eye(2); fix[-1, -1] = np.linalg.det(u @ vt)
    R = u @ fix @ vt; t = mx - my @ R
    return R, t
source_hashes = {str(p.relative_to(SRC)): sha(p) for p in sorted((SRC / "raftup").rglob("*.py"))}


## 3. Run/resume three adjacent pairs

This cell performs the experiments. SCAN-IT cost is generated once per pair and saved as `.npy`; its time is recorded separately from alignment. Existing matching configurations reuse cached cost and alignment. Changing inputs, parameters or source requires a new output directory. All labels are stripped before cost computation and alignment.

In [ ]:
all_scores = []; transforms = {}
for sa, sb in PAIRS:
    a, b = slices[sa], slices[sb]
    folder = OUT / f"{sa}__{sb}"; folder.mkdir(exist_ok=True)
    config = dict(pair=[sa,sb], technology="dlpfc", pixels_per_100um=100, scanit=SCANIT,
                  gpr_radius_um=GPR_RADIUS_UM, source_sha256=source_hashes, manual=MANUAL,
                  inputs={s:{item["filename"]:sha(source_file(s, Path(item["relative_path"]).name))
                           for item in manifest if item["section"]==s} for s in [sa,sb]},
                  annotations_sha256=sha(ANNOTATIONS) if ANNOTATIONS else None,
                  cell_order_A=a.obs_names.tolist(),cell_order_B=b.obs_names.tolist(),genes=a.var_names.tolist())
    config_path = folder / "run_config.json"
    if config_path.exists():
        assert json.loads(config_path.read_text()) == config, "Inputs/source/config changed: choose a new OUT"
    else: write_json(config_path, config)
    aa, bb = a.copy(), b.copy()
    for obj in (aa, bb): obj.obs = obj.obs.iloc[:, :0].copy(); obj.uns.clear()
    timing_path = folder / "timings.json"
    timing = json.loads(timing_path.read_text()) if timing_path.exists() else {}
    cost_path = folder / "feature_cost.npy"
    cached = AUTO_OUT / f"{sa}__{sb}"
    if not cost_path.exists() and (cached / "feature_cost.npy").exists():
        prior = json.loads((cached / "run_config.json").read_text())
        same_inputs = all(prior.get(key) == config.get(key) for key in
                          ("pair", "technology", "pixels_per_100um", "scanit", "inputs", "cell_order_A", "cell_order_B", "genes"))
        old_sources = prior.get("source_sha256", {})
        same_sources = all(source_hashes.get(key) == value for key, value in old_sources.items())
        if same_inputs and same_sources:
            import shutil
            shutil.copy2(cached / "feature_cost.npy", cost_path)
            timing["feature_cost_reused_from"] = str(cached / "feature_cost.npy")
            write_json(timing_path, timing)
            print(sa, sb, "reusing verified SCAN-IT cost", flush=True)
    if not cost_path.exists():
        print(sa, sb, "computing SCAN-IT cost", flush=True)
        started = time.perf_counter()
        cost = compute_scanit_cost(aa, bb, technology="dlpfc", **SCANIT)
        timing["scanit_seconds"] = time.perf_counter()-started
        tmp = folder / "feature_cost.tmp.npy"; np.save(tmp, cost); tmp.replace(cost_path)
        write_json(timing_path, timing)
    else: cost = np.load(cost_path)
    assert cost.shape == (len(a),len(b)) and np.isfinite(cost).all()
    matrix_path = folder / "alignment_matrix.npz"
    if matrix_path.exists():
        P = sparse.load_npz(matrix_path).toarray()
        decision = json.loads((folder / "cutoff_decision.json").read_text())
    else:
        print(sa, sb, "running manual alignment (feature cutoff 0.2)", flush=True)
        started = time.perf_counter()
        try:
            P, decision = run_manual_alignment(aa, bb, cost, technology="dlpfc", pixels_per_100um=100, **MANUAL)
        except RuntimeError as exc:
            if not str(exc).startswith("Recovery rejected:"):
                raise
            timing["alignment_wall_seconds"] = time.perf_counter()-started
            score = dict(pair=f"{sa}__{sb}", status="recovery_rejected", reason=str(exc), **timing)
            write_json(folder / "metrics.json", score)
            write_json(timing_path, timing)
            all_scores.append(score)
            pd.DataFrame(all_scores).to_csv(OUT / "pair_metrics.csv", index=False)
            print(sa, sb, str(exc), flush=True)
            del cost,aa,bb;gc.collect();continue
        timing["alignment_wall_seconds"] = time.perf_counter()-started
        P = np.asarray(P)
        assert P.shape == cost.shape and np.isfinite(P).all() and (P>=0).all()
        write_json(folder / "cutoff_decision.json", decision)
        tmp = folder / "alignment_matrix.tmp.npz"; sparse.save_npz(tmp, sparse.csr_matrix(P)); tmp.replace(matrix_path)
        write_json(timing_path, timing)
    empty = float(P.sum()) <= 0
    score = dict(pair=f"{sa}__{sb}", status="empty_alignment" if empty else "aligned",
                 selected_attempt="manual_fc020", **scores_for(P,a,b), **timing)
    write_json(folder / "metrics.json", score)
    all_scores.append(score)
    pd.DataFrame(all_scores).to_csv(OUT / "pair_metrics.csv", index=False)
    if empty:
        print(sa, sb, "empty alignment under manual cutoffs; no transform/overlay", flush=True)
        del cost,P,aa,bb;gc.collect(); continue
    R,t = rigid_b_to_a(P,a.obsm["spatial"],b.obsm["spatial"])
    transforms[(sa,sb)] = (R,t)
    write_json(folder / "rigid_b_to_a.json", dict(R_row_vector=R.tolist(),t=t.tolist(),units="um"))
    x=a.obsm["spatial"]; y=b.obsm["spatial"]@R+t
    fig,ax=plt.subplots(figsize=(7,7))
    ax.scatter(x[:,0],x[:,1],s=3,alpha=.5,label=sa);ax.scatter(y[:,0],y[:,1],s=3,alpha=.5,label=sb+" aligned")
    ax.set_aspect("equal");ax.invert_yaxis();ax.set_xlabel("X (μm)");ax.set_ylabel("Y (μm)")
    ax.set_title(f"RAFT-UP: {sa}–{sb}");ax.legend()
    fig.savefig(folder / "alignment_overlay.png",dpi=200,bbox_inches="tight");plt.show();plt.close(fig)
    del cost,P,aa,bb;gc.collect()
display(pd.DataFrame(all_scores))

## 4. Four-slice Plotly reconstruction (U2–U5)
Compose the three saved adjacent transforms into the U2 frame. All four slices are displayed together. Z denotes section order only, not physical distance; inter-section spacing remains unverified.


In [ ]:
import plotly.graph_objects as go
fig = go.Figure(); global_R=np.eye(2); global_t=np.zeros(2); coord_rows=[]
for index, section in enumerate(SECTIONS):
    if index:
        pair=OUT/f"{SECTIONS[index-1]}__{section}"/"rigid_b_to_a.json"
        if not pair.exists():
            raise FileNotFoundError(f"Missing transform for {SECTIONS[index-1]}–{section}: {pair}")
        tr=json.loads(pair.read_text());R=np.array(tr["R_row_vector"]);t=np.array(tr["t"])
        global_t=t@global_R+global_t;global_R=R@global_R
    xy=slices[section].obsm["spatial"]@global_R+global_t
    fig.add_trace(go.Scatter3d(x=xy[:,0],y=xy[:,1],z=np.full(len(xy),index),mode="markers",name=section,marker=dict(size=2,opacity=.7)))
    coord_rows.append(pd.DataFrame(dict(section=section,barcode=slices[section].obs_names,x_um=xy[:,0],y_um=xy[:,1],section_index=index)))
pd.concat(coord_rows,ignore_index=True).to_csv(OUT/"reconstruction_coordinates.csv.gz",index=False)
fig.update_layout(title="RAFT-UP: HT206B1 · U2–U5",scene=dict(xaxis_title="X (μm)",yaxis_title="Y (μm)",zaxis=dict(title="Slice order (not physical distance)", tickvals=list(range(len(SECTIONS))), ticktext=SECTIONS),aspectmode="manual",aspectratio=dict(x=1,y=1,z=.7)))
fig.write_html(OUT/"reconstruction_plotly.html",include_plotlyjs=True)
from IPython.display import HTML
display(HTML(fig.to_html(full_html=False, include_plotlyjs=True)))